In [ ]:
# Set parent as root and import config
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
import configs.simulation_config as cfg
from configs.paths import PROCESSED_DIR

import pandas as pd
import numpy as np

In [ ]:
rng = np.random.default_rng(cfg.RANDOM_SEED)

In [ ]:
def assign_transport_mode(nodes, mode_probs):
    """
    Assign transportation mode to each node.
    Mode represents dominant evacuation method.
    """

    modes = list(mode_probs.keys())
    probs = list(mode_probs.values())

    nodes["transport_mode"] = rng.choice(
        modes,
        size=len(nodes),
        p=probs
    )

    return nodes

In [ ]:
def assign_vehicle_occupancy(nodes_modes):
    """
    Assign vehicle occupancy.
    Walkers → 1
    Drivers → random 1–5 people per car
    """

    # Default walkers = 1
    nodes_modes["vehicle_occ"] = 1

    drive_mask = nodes_modes["transport_mode"] == "drive"

    nodes_modes.loc[drive_mask, "vehicle_occ"] = rng.integers(
        1,
        6,
        size=drive_mask.sum()
    )

    return nodes_modes

In [ ]:
nodes = pd.read_csv(
    PROCESSED_DIR / "hatyai_nodes_flood_multi.csv"
)

nodes_modes = assign_transport_mode(nodes, cfg.TRANSPORTATION_MODE_PROBS)
nodes_modes = assign_vehicle_occupancy(nodes_modes)

nodes_modes.to_csv(
    PROCESSED_DIR / "hatyai_nodes_transport.csv",
    index=False
)